# 02 — Preprocessing
Cleaning, encoding, and feature engineering, driven entirely by
`src/preprocessing.py` and `src/feature_engineering.py`. Output is
saved to `data/processed/` for reuse by every downstream notebook.

In [7]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display

from src.preprocessing import (
    rename_columns, treat_skip_logic_missing_values, harmonize_categories,
    check_mapping_coverage, remove_duplicates, encode, validate
)
from src.feature_engineering import (
    engineer_features, ANALYSIS_VARIABLES, FINAL_COLUMNS, CONSTRUCTS,
    TARGET_COLUMN, CANDIDATE_FEATURES, COMPOSITE_VARS
)

## Load and select analysis variables

In [8]:
df = pd.read_excel(PROJECT_ROOT / "data" / "raw" / "tracer_study_raw.xlsx")
df = rename_columns(df)

missing_cols = [c for c in ANALYSIS_VARIABLES if c not in df.columns]
print(f"Missing analysis variables: {missing_cols if missing_cols else 'none'}")

df_analysis = df[ANALYSIS_VARIABLES].copy()
print(f"Analysis dataset shape: {df_analysis.shape}")

Missing analysis variables: none
Analysis dataset shape: (157, 59)


## Mapping coverage check

Run after skip-logic fill and harmonization, before encoding — flags
any raw category value that won't match its mapping (typos, casing,
off-by-one numbering) so it doesn't silently become `NaN` during
encoding with no warning at the time.

In [9]:
df_check = treat_skip_logic_missing_values(df_analysis)
df_check = harmonize_categories(df_check)
check_mapping_coverage(df_check)

All checked columns fully covered by their mappings.


## Run the full pipeline

In [10]:
df_analysis = treat_skip_logic_missing_values(df_analysis)
df_analysis = harmonize_categories(df_analysis)
df_analysis = remove_duplicates(df_analysis)
df_analysis = encode(df_analysis)
df_analysis = engineer_features(df_analysis, CONSTRUCTS)

df_final = df_analysis[FINAL_COLUMNS].copy()
validate(df_final);

Shape: (157, 12)
Missing (total): 0
Missing (by column): none
Duplicate rows: 0


## Validation checks

In [11]:
print("Target distribution:")
display(df_final[TARGET_COLUMN].value_counts().to_frame("count"))

print("\nAge range (birth-year correction check):")
display(df_final["age"].describe())
assert df_final["age"].min() >= 18, "Implausible age detected — check birth_year data entry."

print("\nComposite index ranges (all 7, incl. the 2 excluded from modeling):")
# df_analysis (not df_final) — df_final only keeps CANDIDATE_FEATURES,
# which excludes employability_index/retrospective_choice_index as
# leakage risks (see README "Feature set"). They're still computed on
# df_analysis for descriptive/EDA purposes, just not modeled.
display(df_analysis[COMPOSITE_VARS].describe().T[["min", "max", "mean"]])

print(f"\nModeling feature count: {len(CANDIDATE_FEATURES)}")
assert df_final.isnull().sum().sum() == 0, "Unexpected missing values after preprocessing."
print("\nAll validation checks passed.")

Target distribution:


,count
employment_status,
1,126
0,31



Age range (birth-year correction check):


count    157.000000
mean      23.656051
std        2.138515
min       22.000000
25%       23.000000
50%       23.000000
75%       24.000000
max       36.000000
Name: age, dtype: float64


Composite index ranges (all 7, incl. the 2 excluded from modeling):


,min,max,mean
school_choice_index,1.000000,5.0,4.673885
learning_experience_index,3.000000,5.0,4.396178
study_conditions_index,1.777778,5.0,4.313517
guidance_support_index,2.000000,5.0,4.498408
employability_index,2.000000,5.0,4.575796
competency_index,2.000000,5.0,4.539490
retrospective_choice_index,2.000000,5.0,4.547771



Modeling feature count: 11

All validation checks passed.


## Save processed dataset

In [12]:
output_path = PROJECT_ROOT / "data" / "processed" / "tracer_study_processed.csv"
df_final.to_csv(output_path, index=False)
print(f"Saved: {output_path.relative_to(PROJECT_ROOT)} ({df_final.shape[0]} rows, {df_final.shape[1]} columns)")

Saved: data/processed/tracer_study_processed.csv (157 rows, 12 columns)
